# 8.04 CosmosDBSaver — Azure Cosmos DB (NoSQL) 체크포인터

`langgraph-checkpoint-cosmosdb` 는 커뮤니티 패키지로, Azure Cosmos DB **NoSQL API** 컨테이너 위에 LangGraph 체크포인트를 저장한다.
DynamoDB saver 를 참고해 만들어졌으며 key 구조는 `partition_key = checkpoint$<thread_id>$<ns>$` + `id = checkpoint$<thread_id>$<ns>$<checkpoint_id>`.

## 학습 목표

- Cosmos 계정 · 데이터베이스 · 컨테이너 준비 (키 기반 인증)
- `CosmosDBSaver(database_name, container_name)` 로 인스턴스 생성 — 필요한 리소스는 자동 create_if_not_exists
- thread_id 기반 복원 · `get_state_history` 동작 확인
- 같은 클래스의 `aget_tuple` / `aput` 등 비동기 메서드로 async 워크플로우 연결

## 언제 쓰나

- 이미 Azure 에서 운영 중 — LangGraph 만 추가하고 데이터는 Cosmos 로 몰아두고 싶을 때
- 글로벌 배포 · 지역 복제가 필요해 Cosmos 의 multi-region 을 쓰고 싶을 때
- **서버리스** 가격 정책으로 저 TPS 워크로드를 관리형으로 돌리고 싶을 때

⚠️ 로컬 실행이 어렵다. Azure Cosmos DB **emulator** 또는 무료 티어 계정을 미리 준비해야 한다. 키가 없으면 probe 에서 바로 멈춘다.


## 8.04.1 환경 설정 · 서비스 연결

`.env` 에 아래 두 값을 미리 채운다. 커뮤니티 패키지 구현이 이 정확한 이름을 읽는다:

```
COSMOSDB_ENDPOINT=https://<account>.documents.azure.com:443/
COSMOSDB_KEY=<primary-or-secondary-key>
```

probe 셀은 두 환경변수 존재 + `CosmosClient` 생성 + `list_databases()` 한 번을 실제로 돌려 연결을 확인한다. 실패 시 `azure.cosmos.exceptions` 에서 예외가 올라오고 이후 셀은 실행되지 않는다.

> Azure Cosmos DB emulator 를 쓸 때도 동일한 두 변수를 쓴다 (`https://localhost:8081/` + emulator 의 고정 key).


In [ ]:
# !pip install -U langgraph langgraph-checkpoint-cosmosdb azure-cosmos langchain langchain-openai python-dotenv

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"
assert os.environ.get("COSMOSDB_ENDPOINT"), "COSMOSDB_ENDPOINT 필요 (.env)"
assert os.environ.get("COSMOSDB_KEY"),      "COSMOSDB_KEY 필요 (.env)"

from azure.cosmos import CosmosClient
# 실제 계정 접근 확인 — 자격/네트워크 실패 시 여기서 예외
_client = CosmosClient(os.environ["COSMOSDB_ENDPOINT"], os.environ["COSMOSDB_KEY"])
list(_client.list_databases())
print("Cosmos reachable:", os.environ["COSMOSDB_ENDPOINT"])

from langgraph_checkpoint_cosmosdb import CosmosDBSaver
print("class:", CosmosDBSaver.__name__)


## 8.04.2 기본 사용법 — `CosmosDBSaver(database_name, container_name)`

생성자는 database / container 이름 두 개만 받는다. 내부에서 `create_database_if_not_exists` → `create_container_if_not_exists(partition_key=/partition_key)` 순으로 리소스를 확보한다. 따라서 **별도 `.setup()` 이 없고** 첫 인스턴스 생성이 곧 셋업이다.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turn: Annotated[int, operator.add]

def respond(state: ChatState) -> ChatState:
    last = state["messages"][-1].content if state["messages"] else "(empty)"
    next_turn = state.get("turn", 0) + 1
    return {
        "messages": [{"role": "assistant", "content": f"echo[{next_turn}]: {last}"}],
        "turn": 1,
    }

builder = StateGraph(ChatState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

saver = CosmosDBSaver(database_name="langgraph", container_name="checkpoints_demo")
graph = builder.compile(checkpointer=saver)
print("compiled:", graph.checkpointer.__class__.__name__)


## 8.04.3 thread_id 스코프와 복원

Cosmos 의 partition_key 는 `checkpoint$<thread_id>$<ns>$` 로 구성되므로 **thread 단위 쓰기/읽기가 단일 파티션** 에 떨어진다 (읽기 지연 · RU 비용 최적). 여러 프로세스가 같은 계정에 붙어 같은 thread_id 를 공유할 수 있다.

In [ ]:
cfg = {"configurable": {"thread_id": "cosmos-demo-1"}}
for msg in ["첫 메시지", "두 번째", "세 번째"]:
    out = graph.invoke({"messages": [{"role": "user", "content": msg}]}, cfg)
print("turn:", out["turn"])

# 새 saver 인스턴스로 같은 컨테이너를 다시 연다 (프로세스 재시작과 유사)
saver2 = CosmosDBSaver(database_name="langgraph", container_name="checkpoints_demo")
graph2 = builder.compile(checkpointer=saver2)
snap = graph2.get_state(cfg)
print("재오픈 후 turn =", snap.values.get("turn"))
print("messages 수 =", len(snap.values.get("messages", [])))


## 8.04.4 `get_state_history` · time travel

Cosmos 구현체는 `list` · `get_tuple` · `put_writes` 를 지원하므로 time travel API 가 그대로 동작한다. 과거 체크포인트로 분기하면 새 체크포인트가 동일 파티션에 추가된다.

In [ ]:
history = list(graph.get_state_history(cfg))
print(f"체크포인트 수: {len(history)}")
for i, h in enumerate(history[:5]):
    print(f"  [{i}] turn={h.values.get('turn')} next={h.next}")

# turn==1 시점으로 fork
target = next(h for h in reversed(history) if h.values.get("turn") == 1 and h.next == ())
branched = graph.invoke(
    {"messages": [{"role": "user", "content": "fork at turn 1"}]},
    target.config,
)
print("\nbranched turn =", branched["turn"])
print("latest(branch 후) turn =", graph.get_state(cfg).values["turn"])
print("전체 history 길이 =", len(list(graph.get_state_history(cfg))))


## 8.04.5 비동기 메서드 사용 — `aget_tuple` · `aput`

`langgraph-checkpoint-cosmosdb==0.2.x` 는 별도 `AsyncCosmosDBSaver` 클래스를 노출하지 않고, 같은 `CosmosDBSaver` 인스턴스에 `aget_tuple` · `aput` · `aput_writes` · `alist` · `adelete` 를 제공한다. 내부적으로 `asyncio.to_thread` 로 동기 메서드를 감싼다. 그래프 레벨에서 `ainvoke` · `aget_state` 를 그대로 쓸 수 있다.

In [ ]:
import asyncio

async def run_async_demo():
    agraph = builder.compile(checkpointer=saver)   # 같은 saver 재사용
    cfg_a = {"configurable": {"thread_id": "cosmos-async-1"}}
    for msg in ["async 1", "async 2"]:
        await agraph.ainvoke(
            {"messages": [{"role": "user", "content": msg}]},
            cfg_a,
        )
    snap = await agraph.aget_state(cfg_a)
    print("async turn =", snap.values["turn"])

await run_async_demo()


## 8.04.6 `create_agent` 에서 Cosmos 메모리

Azure 위에 올린 에이전트의 멀티턴 메모리 경로.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    tools=[],
    checkpointer=CosmosDBSaver(database_name="langgraph", container_name="agent_checkpoints"),
    system_prompt="너는 짧게 답하는 메모리 에이전트야.",
)

cfg = {"configurable": {"thread_id": "agent-cosmos-1"}}
r1 = agent.invoke({"messages": [{"role": "user", "content": "나는 부산 사람이야"}]}, cfg)
print("1:", r1["messages"][-1].content[:120])
r2 = agent.invoke({"messages": [{"role": "user", "content": "내가 어디 살았지?"}]}, cfg)
print("2:", r2["messages"][-1].content[:120])


## 체크리스트

- [ ] 환경변수는 정확히 `COSMOSDB_ENDPOINT` · `COSMOSDB_KEY` (패키지 구현이 읽는 이름)
- [ ] 컨테이너 partition_key 는 `/partition_key` 로 자동 생성 — **직접 만들지 말 것**
- [ ] 서버리스/프로비저닝 RU 설정은 Azure Portal · Bicep 에서 별도 관리
- [ ] Cosmos emulator 로 로컬 개발 가능 (HTTPS 인증서만 신뢰 설정 필요)
- [ ] 별도 `AsyncCosmosDBSaver` 클래스 없음 — 같은 인스턴스의 `a*` 메서드 사용

## 다음

- `09_stores/` — 스레드 간 공유 지식 (`BaseStore`) 계층
- `docs/skills/langgraph-persistence.md` — subgraph 스코프 · `checkpoint_ns` 내부 구조
- Postgres 로 옮기고 싶다면 `03_postgres.ipynb` 의 마이그레이션 패턴 적용

## 참고

- `langgraph-checkpoint-cosmosdb`: https://pypi.org/project/langgraph-checkpoint-cosmosdb/
- Azure Cosmos DB 무료 티어 · emulator: https://learn.microsoft.com/azure/cosmos-db/
- Azure SDK `azure-cosmos`: https://pypi.org/project/azure-cosmos/
